# Get the libraries and data loaded

In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# Plot style: seaborn look without gridlines
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = False
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

category_colors = {
    "FOODS": "tab:blue",
    "HOBBIES": "tab:orange",
    "HOUSEHOLD": "tab:green",
}

# Importing data
# I am uing the Walmart dataset from the M5 firecasting competition, which is available in the `datasetsforecast` package.
from datasetsforecast.m5 import M5

## Calendar and Event Features

The M5 dataset includes calendar-based features that help explain demand patterns.

### Event columns

- `event_name_1`: Name of the first event or holiday on the date.
- `event_type_1`: Type of the first event, such as National, Religious, Sporting, or Cultural.
- `event_name_2`: Name of a second event if two events happen on the same date.
- `event_type_2`: Type of the second event.

These features are useful because holidays and special events can create unusual demand patterns, such as sharp drops on Christmas or changes around Thanksgiving.

### SNAP columns

- `snap_CA`: Binary indicator for whether SNAP purchases are active/allowed in California on that date.
- `snap_TX`: Binary indicator for whether SNAP purchases are active/allowed in Texas on that date.
- `snap_WI`: Binary indicator for whether SNAP purchases are active/allowed in Wisconsin on that date.

SNAP stands for Supplemental Nutrition Assistance Program. In this dataset, the SNAP columns do not tell us who used SNAP or how much was purchased with SNAP. They only indicate whether SNAP purchasing was active for that state on that date.

These columns may help explain demand changes, especially in categories like FOODS.

In [12]:
help(M5.load)

Help on function load in module datasetsforecast.m5:

load(directory: str, cache: bool = True) -> Tuple[pandas.core.frame.DataFrame, pandas.core.frame.DataFrame, pandas.core.frame.DataFrame]
    Downloads and loads M5 data.

    Args:
        directory (str): Directory where data will be downloaded.
        cache (bool): If `True` saves and loads.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
            Target time series with columns ['unique_id', 'ds', 'y'],
            Exogenous time series with columns ['unique_id', 'ds', 'y'],
            Static exogenous variables with columns ['unique_id', 'ds']
            and static variables.



# Audit the dataset

In [4]:
Y_df, X_df, S_df = M5.load(directory="data/m5")

print("Y_df:", Y_df.shape)
print("X_df:", X_df.shape)
print("S_df:", S_df.shape)

Y_df: (47649940, 3)
X_df: (47649940, 10)
S_df: (30490, 6)


In [9]:
# target time series
Y_df.head()

,unique_id,ds,y
0,FOODS_1_001_CA_1,2011-01-29,3.00
1,FOODS_1_001_CA_1,2011-01-30,0.00
2,FOODS_1_001_CA_1,2011-01-31,0.00
3,FOODS_1_001_CA_1,2011-02-01,1.00
4,FOODS_1_001_CA_1,2011-02-02,4.00


In [15]:
# exogenous time series
X_df.head()

,unique_id,ds,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1,2011-01-29,nan,nan,nan,nan,0,0,0,2.00
1,FOODS_1_001_CA_1,2011-01-30,nan,nan,nan,nan,0,0,0,2.00
2,FOODS_1_001_CA_1,2011-01-31,nan,nan,nan,nan,0,0,0,2.00
3,FOODS_1_001_CA_1,2011-02-01,nan,nan,nan,nan,1,1,0,2.00
4,FOODS_1_001_CA_1,2011-02-02,nan,nan,nan,nan,1,0,1,2.00


In [13]:
#static time series
S_df.head()

,unique_id,item_id,dept_id,cat_id,store_id,state_id
0,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
1969,FOODS_1_001_CA_2,FOODS_1_001,FOODS_1,FOODS,CA_2,CA
3938,FOODS_1_001_CA_3,FOODS_1_001,FOODS_1,FOODS,CA_3,CA
5907,FOODS_1_001_CA_4,FOODS_1_001,FOODS_1,FOODS,CA_4,CA
7875,FOODS_1_001_TX_1,FOODS_1_001,FOODS_1,FOODS,TX_1,TX


In [14]:
for name, df in {"Y_df": Y_df, "X_df": X_df, "S_df": S_df}.items():
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print(df.dtypes)


Y_df
shape: (47649940, 3)
columns: ['unique_id', 'ds', 'y']
unique_id          category
ds           datetime64[ns]
y                   float32
dtype: object

X_df
shape: (47649940, 10)
columns: ['unique_id', 'ds', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']
unique_id             category
ds              datetime64[ns]
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                  uint8
snap_TX                  uint8
snap_WI                  uint8
sell_price             float32
dtype: object

S_df
shape: (30490, 6)
columns: ['unique_id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
unique_id    category
item_id      category
dept_id      category
cat_id       category
store_id     category
state_id     category
dtype: object


In [16]:
print("Date range:", Y_df["ds"].min(), "to", Y_df["ds"].max())
print("Number of unique series:", Y_df["unique_id"].nunique())
print("Rows:", len(Y_df))
print("Duplicate unique_id-date rows:", Y_df.duplicated(subset=["unique_id", "ds"]).sum())
print("Negative sales rows:", (Y_df["y"] < 0).sum())
print("Missing target values:", Y_df["y"].isna().sum())

Date range: 2011-01-29 00:00:00 to 2016-06-19 00:00:00
Number of unique series: 30490
Rows: 47649940
Duplicate unique_id-date rows: 0
Negative sales rows: 0
Missing target values: 0


In [17]:
m5 = (
    Y_df
    .merge(X_df, on=["unique_id", "ds"], how="left")
    .merge(S_df, on="unique_id", how="left")
)

print("m5 shape:", m5.shape)
m5.head()

m5 shape: (47649940, 16)


,unique_id,ds,y,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,item_id,dept_id,cat_id,store_id,state_id
0,FOODS_1_001_CA_1,2011-01-29,3.00,nan,nan,nan,nan,0,0,0,2.00,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
1,FOODS_1_001_CA_1,2011-01-30,0.00,nan,nan,nan,nan,0,0,0,2.00,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
2,FOODS_1_001_CA_1,2011-01-31,0.00,nan,nan,nan,nan,0,0,0,2.00,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
3,FOODS_1_001_CA_1,2011-02-01,1.00,nan,nan,nan,nan,1,1,0,2.00,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
4,FOODS_1_001_CA_1,2011-02-02,4.00,nan,nan,nan,nan,1,0,1,2.00,FOODS_1_001,FOODS_1,FOODS,CA_1,CA


In [21]:
missing_summary = (
    m5.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary['missing_rate'].sum()

np.float64(0.0)

In [22]:
missing_summary = (
    m5.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary.head(25)

,column,missing_rate
0,unique_id,0.00
1,ds,0.00
2,y,0.00
3,event_name_1,0.00
4,event_type_1,0.00
5,event_name_2,0.00
6,event_type_2,0.00
7,snap_CA,0.00
8,snap_TX,0.00
9,snap_WI,0.00


In [19]:
series_lengths = (
    m5.groupby("unique_id", observed=True)["ds"]
    .agg(start_date="min", end_date="max", observations="count")
    .reset_index()
)

series_lengths.describe(include="all")

,unique_id,start_date,end_date,observations
count,30490,30490,30490,"30,490.00"
unique,30490,NaN,NaN,NaN
top,FOODS_1_001_CA_1,NaN,NaN,NaN
freq,1,NaN,NaN,NaN
mean,NaN,2012-03-10 04:40:03.935716608,2016-06-19 00:00:00.000000256,"1,562.81"
min,NaN,2011-01-29 00:00:00,2016-06-19 00:00:00,124.00
25%,NaN,2011-01-30 00:00:00,2016-06-19 00:00:00,"1,203.00"
50%,NaN,2011-07-07 00:00:00,2016-06-19 00:00:00,"1,810.00"
75%,NaN,2013-03-05 00:00:00,2016-06-19 00:00:00,"1,968.00"
max,NaN,2016-02-17 00:00:00,2016-06-19 00:00:00,"1,969.00"
